In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import plotly.graph_objects as go

# Database Connection
engine = create_engine('postgresql://postgres:""@localhost:5432/profit_optimization')
df_master = pd.read_sql("SELECT * FROM v_master_profitability", engine)



In [ ]:
def display_kpi_table(df):
    total_gross = df['gross_revenue_usd'].sum()
    total_net = df['net_revenue_usd'].sum()
    total_profit = df['net_profit_usd'].sum()
    return_rate = df['is_returned'].mean() * 100
    avg_margin = (total_profit / total_net) * 100 if total_net != 0 else 0

    kpi_df = pd.DataFrame({
        'Metric': ['Total Gross Revenue', 'Total Net Revenue', 'Total Net Profit', 'Overall Return Rate (%)', 'Average Profit Margin (%)'],
        'Value': [total_gross, total_net, total_profit, return_rate, avg_margin]
    })
    kpi_df['Value'] = kpi_df['Value'].apply(lambda x: f"${x:,.2f}" if 'Rate' not in str(x) else f"{x:.2f}%")
    return kpi_df

kpi_table = display_kpi_table(df_master)
print(kpi_table)

def plot_baseline_waterfall(df):
    total_gross = df['gross_revenue_usd'].sum()
    cogs = df['cogs'].sum()
    returns_loss = total_gross - df['net_revenue_usd'].sum()
    shipping = df['total_shipping_cost'].sum()
    marketing = df['total_marketing_cost'].sum()
    labor = df['labor_cost'].sum()
    net_profit = df['net_profit_usd'].sum()

    fig = go.Figure(go.Waterfall(
        name="Baseline Profitability",
        orientation="v",
        measure=["relative","relative","relative","relative","relative","relative","total"],
        x=["Gross Revenue","COGS","Returns Loss","Shipping Costs","Marketing Spend","Labor Costs","Net Profit"],
        y=[total_gross, -cogs, -returns_loss, -shipping, -marketing, -labor, net_profit],
        text=[f"${v/1e6:.1f}M" for v in [total_gross, cogs, returns_loss, shipping, marketing, labor, net_profit]],
        textposition="outside",
        connector={"line":{"color":"rgb(63, 63, 63)"}},
        decreasing={"marker":{"color":"red"}},
        increasing={"marker":{"color":"green"}},
        totals={"marker":{"color":"darkblue"}}
    ))

    fig.update_layout(title="Baseline Profit Waterfall (USD)", showlegend=False)
    fig.show()

plot_baseline_waterfall(df_master)

                      Metric              Value
0        Total Gross Revenue  $1,980,857,202.94
1          Total Net Revenue  $1,802,268,864.68
2           Total Net Profit   $-617,824,994.77
3    Overall Return Rate (%)             $17.81
4  Average Profit Margin (%)            $-34.28


Global Omnichannel E-Commerce Profit Challenge

Overview:
Our company operates in Turkey, Germany, and the UAE across D2C and B2B channels. Despite strong revenue growth, net profit is declining significantly, highlighting challenges in converting top-line growth into sustainable profitability.

Key Issues:

High D2C Return Rates: 17.8% return rate significantly erodes net revenue and profit.
COGS Volatility Due to FX Exposure: Currency fluctuations make cost planning unpredictable.
Inefficient Marketing Allocation: Marketing ROI is low, impacting overall profitability.
Long B2B Payment Cycles: Delays in cash collection constrain financial flexibility.

Financial Impact / KPIs:

| Metric                | Value             |
| --------------------- | ----------------- |
| Total Gross Revenue   | $1,980,857,202.94 |
| Total Net Revenue     | $1,802,268,864.68 |
| Total Net Profit      | $-617,824,994.77  |
| Overall Return Rate   | 17.81%            |
| Average Profit Margin | -34.28%           |


Insights:

Despite nearly $2B in gross revenue, the company is currently experiencing a $618M net loss.
The average profit margin of -34.28% means the company loses $0.34 for every $1 in sales.
High return rates and volatile costs are the primary drivers of financial underperformance.



KPI Table  summarizes revenue, profit, return rates, and margins.
Baseline Waterfall Chart shows how gross revenue is eroded by costs, returns, and expenses to arrive at net profit.

 Notes:
“This slide establishes the magnitude of the problem and its financial impact. It sets the stage for the next steps: analyzing loss drivers and simulating strategic interventions to optimize profitability.”

In [3]:
import plotly.express as px

def generate_profit_matrix(df):
    """
    Segment x Channel x Country bazlı kâr ve kâr marjını hesaplar.
    """
    matrix = df.groupby(['country', 'channel', 'segment']).agg(
        order_count=('order_id', 'count'),
        avg_return_rate=('is_returned', 'mean'),
        total_profit_usd=('net_profit_usd', 'sum')
    ).reset_index()

    revenue_agg = df.groupby(['country', 'channel', 'segment'])['net_revenue_usd'].sum().reset_index()
    matrix = matrix.merge(revenue_agg, on=['country', 'channel', 'segment'])

    
    matrix['profit_margin_pct'] = (matrix['total_profit_usd'] / matrix['net_revenue_usd']) * 100

    
    matrix['profit_per_order'] = matrix['total_profit_usd'] / matrix['order_count']

    return matrix.sort_values(by='profit_margin_pct', ascending=False)

profit_matrix = generate_profit_matrix(df_master)

def plot_profit_margin_heatmap(matrix):
    fig = px.density_heatmap(
        matrix,
        x='segment',
        y='country',
        z='profit_margin_pct',
        facet_col='channel',
        color_continuous_scale='RdYlGn',
        title="Profit Margin % by Country, Segment, and Channel",
        labels={'profit_margin_pct':'Profit Margin (%)'}
    )
    fig.show()

def plot_profit_per_order_heatmap(matrix):
    fig = px.density_heatmap(
        matrix,
        x='segment',
        y='country',
        z='profit_per_order',
        facet_col='channel',
        color_continuous_scale='RdYlGn',
        title="Profit per Order by Country, Segment, and Channel",
        labels={'profit_per_order':'Profit per Order (USD)'}
    )
    fig.show()

plot_profit_margin_heatmap(profit_matrix)
plot_profit_per_order_heatmap(profit_matrix)

Cause Analysis / Segment Performance

Segment-Wise Profitability Insights

Presentation:
Which country × channel × segment combinations are profitable, and which are loss-making? Segment-level analysis identifies the main sources of financial underperformance and guides corrective strategy.



B2B Premium and D2C Premium segments are the most profitable (dark green in heatmap).
UAE D2C Entry is the largest loss contributor (dark red).
Segment-specific return rates and profit margins reveal the primary drivers of losses, guiding targeted interventions.




In [4]:
np.random.seed(42) 
def simulate_scenario(df, mkt_change=0.0, return_improve=0.0, shipping_cost_shock=0.0):
    sim_df = df.copy()

    sim_df['total_marketing_cost'] *= (1 + mkt_change)
    
    segment_impact = {'Entry': 0.3, 'Mid': 0.2, 'Premium': 0.1}
    for segment, impact in segment_impact.items():
        seg_mask = sim_df['segment'] == segment
        returned_idx = sim_df[seg_mask & sim_df['is_returned']].index
        fix_count = int(len(returned_idx) * return_improve * impact)
        if fix_count > 0:
            fix_idx = np.random.choice(returned_idx, fix_count, replace=False)
            sim_df.loc[fix_idx, 'is_returned'] = False

    sim_df['net_revenue_usd'] = np.where(sim_df['is_returned'], 0, sim_df['gross_revenue_usd'])

    sim_df['total_shipping_cost'] *= (1 + shipping_cost_shock)

    sim_df['cogs'] = sim_df['base_price_usd'] * sim_df['quantity']
    sim_df['labor_cost'] = sim_df['gross_revenue_usd'] * sim_df['labor_ratio']

    sim_df['new_net_profit'] = sim_df['net_revenue_usd'] - sim_df['cogs'] - sim_df['total_shipping_cost'] - sim_df['total_marketing_cost'] - sim_df['labor_cost']
    
    return sim_df

simulated_df = simulate_scenario(df_master, mkt_change=0.05, return_improve=0.20, shipping_cost_shock=0.10)

def plot_waterfall_comparison(df_baseline, df_simulated):
    categories = ["Gross Revenue","COGS","Returns Loss","Shipping","Marketing","Labor","Net Profit"]
    baseline_vals = [
        df_baseline['gross_revenue_usd'].sum(),
        df_baseline['cogs'].sum(),
        df_baseline['gross_revenue_usd'].sum() - df_baseline['net_revenue_usd'].sum(),
        df_baseline['total_shipping_cost'].sum(),
        df_baseline['total_marketing_cost'].sum(),
        df_baseline['labor_cost'].sum(),
        df_baseline['net_profit_usd'].sum()
    ]
    sim_vals = [
        df_simulated['gross_revenue_usd'].sum(),
        df_simulated['cogs'].sum(),
        df_simulated['gross_revenue_usd'].sum() - df_simulated['net_revenue_usd'].sum(),
        df_simulated['total_shipping_cost'].sum(),
        df_simulated['total_marketing_cost'].sum(),
        df_simulated['labor_cost'].sum(),
        df_simulated['new_net_profit'].sum()
    ]
    
    fig = go.Figure()
    fig.add_trace(go.Waterfall(
        name="Baseline",
        x=categories,
        y=[v if i==0 or i==6 else -v for i,v in enumerate(baseline_vals)],
        measure=["relative"]*6 + ["total"],
        text=[f"${v/1e6:.1f}M" for v in baseline_vals],
        textposition="outside",
        decreasing={"marker":{"color":"red"}},
        increasing={"marker":{"color":"green"}},
        totals={"marker":{"color":"darkblue"}}
    ))
    fig.add_trace(go.Waterfall(
        name="Simulated",
        x=categories,
        y=[v if i==0 or i==6 else -v for i,v in enumerate(sim_vals)],
        measure=["relative"]*6 + ["total"],
        text=[f"${v/1e6:.1f}M" for v in sim_vals],
        textposition="outside",
        decreasing={"marker":{"color":"orange"}},
        increasing={"marker":{"color":"lightgreen"}},
        totals={"marker":{"color":"darkgreen"}}
    ))
    fig.update_layout(title="Baseline vs Simulated Profit Waterfall", waterfallgap=0.5)
    fig.show()

plot_waterfall_comparison(df_master, simulated_df)

def plot_segment_profit_shift(df_baseline, df_simulated):
    seg_base = df_baseline.groupby('segment')['net_profit_usd'].sum().reset_index()
    seg_sim = df_simulated.groupby('segment')['new_net_profit'].sum().reset_index()
    seg_comp = seg_base.merge(seg_sim, on='segment')
    seg_comp.columns = ['Segment','Baseline','Simulated']
    seg_comp['Profit Change'] = seg_comp['Simulated'] - seg_comp['Baseline']
    
    fig = px.bar(
        seg_comp,
        x='Segment',
        y='Profit Change',
        color='Profit Change',
        color_continuous_scale='RdYlGn',
        title="Segment-wise Profit Change: Simulated vs Baseline",
        labels={'Profit Change':'Δ Net Profit (USD)'}
    )
    fig.show()

plot_segment_profit_shift(df_master, simulated_df)

simulated_df = simulate_scenario(
    df_master,
    mkt_change=0.05,        
    return_improve=0.20,    
    shipping_cost_shock=0.10 
)

def simulation_summary(df_baseline, df_simulated):
    baseline_profit = df_baseline['net_profit_usd'].sum()
    simulated_profit = df_simulated['new_net_profit'].sum()
    profit_change = simulated_profit - baseline_profit
    profit_change_pct = (profit_change / abs(baseline_profit)) * 100
    
    summary_df = pd.DataFrame({
        "Metric": ["Baseline Profit", "Simulated Profit", "Profit Change", "Profit Change (%)"],
        "Value": [baseline_profit, simulated_profit, profit_change, profit_change_pct]
    })
    summary_df['Value'] = summary_df['Value'].apply(lambda x: f"${x:,.2f}" if abs(x) > 1 else f"{x:.2f}%")
    return summary_df

sim_summary = simulation_summary(df_master, simulated_df)
print(sim_summary)

              Metric             Value
0    Baseline Profit  $-617,824,994.77
1   Simulated Profit  $-631,417,550.01
2      Profit Change   $-13,592,555.24
3  Profit Change (%)            $-2.20


Simulation / What-If Analysis

How would net profit change if we adjust key levers like marketing spend, return reduction, or shipping costs?
We ran a scenario simulation to quantify the impact of these strategic actions on overall profitability.

Baseline vs Simulated Waterfall Chart: Shows how gross revenue is impacted by COGS, returns, shipping, marketing, and labor to arrive at net profit under the simulated scenario.

Premium segments (both D2C and B2B) are the largest contributors to losses in the simulation, followed by Mid and then Entry segments.
Returns and shipping costs are the most sensitive levers: controlling returns and optimizing shipping costs can materially reduce losses.
Even modest interventions (e.g., 20% return reduction, 10% shipping increase, 5% marketing spend increase) only slightly improve profitability, highlighting the scale of the underlying cost and margin challenges.
Strategic focus should be on Premium segments and return/COGS management to maximize profit stabilization.



In [ ]:
simulated_df = simulate_scenario(
    df_master,
    mkt_change=0.05,
    return_improve=0.20,
    shipping_cost_shock=0.10
)

new_margin = simulated_df['new_net_profit'].sum() / simulated_df['net_revenue_usd'].sum()
profit_change = simulated_df['new_net_profit'].sum() - simulated_df['net_profit_usd'].sum()
best_segment_row = simulated_df.groupby(['country','channel','segment'])['new_net_profit'].sum().idxmax()
best_segment = f"{best_segment_row[0]} {best_segment_row[1]} {best_segment_row[2]}"

decision_table = pd.DataFrame({
    'Metric': ['New Net Margin', 'Profit Change', 'Best Segment'],
    'Value': [
        f"{new_margin*100:.2f}%", 
        f"${profit_change:,.2f}", 
        best_segment
    ]
})

print(decision_table)

           Metric            Value
0  New Net Margin          -34.89%
1   Profit Change  $-13,602,320.99
2    Best Segment    UAE D2C Entry


Best Segment / Recommended Strategy

Based on the simulation results, the segment with the least loss and highest potential for targeted interventions is UAE D2C Entry.
This segment is suitable for marketing optimization and return rate improvement to stabilize net profit.

Strategic action:
Optimize marketing allocation in this segment.
Focus on reducing return rates to protect net revenue.
Impact: Targeted actions here will minimize losses and maximize the efficiency of investment.
Portfolio value: Shows a concrete, actionable insight for business stakeholders, making the project look like a product-ready data solution.